In [ ]:
!pip install music21 tensorflow numpy

In [ ]:
!git clone https://github.com/jukedeck/nottingham-dataset.git

Cloning into 'nottingham-dataset'...
remote: Enumerating objects: 3119, done.
remote: Total 3119 (delta 0), reused 0 (delta 0), pack-reused 3119 (from 1)
Receiving objects: 100% (3119/3119), 879.17 KiB | 5.90 MiB/s, done.
Resolving deltas: 100% (1432/1432), done.


In [ ]:
import os
from music21 import converter, instrument, note, chord

dataset_path = "nottingham-dataset"

midi_files = []

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".mid") or file.endswith(".midi"):
            midi_files.append(os.path.join(root, file))

print("Total MIDI files found:", len(midi_files))
print("First 5 files:")
for file in midi_files[:5]:
    print(file)

Total MIDI files found: 3089
First 5 files:
nottingham-dataset/MIDI/reelsa-c28.mid
nottingham-dataset/MIDI/jigs117.mid
nottingham-dataset/MIDI/jigs70.mid
nottingham-dataset/MIDI/waltzes3.mid
nottingham-dataset/MIDI/reelsa-c50.mid


In [ ]:
from music21 import converter, note, chord

notes = []

for file in midi_files[:100]:
    try:
        score = converter.parse(file)

        for element in score.flatten().notes:
            if isinstance(element, note.Note):
                notes.append(str(element.pitch))
            elif isinstance(element, chord.Chord):
                notes.append('.'.join(str(n) for n in element.normalOrder))

    except Exception as e:
        pass

print("Total note sequences collected:", len(notes))
print("First 30 notes:")
print(notes[:30])

Total note sequences collected: 27455
First 30 notes:
['G3', '0.4.7', 'A3', 'C4', 'D4', 'E4', 'G4', 'A4', 'B4', 'C5', '0.4.7', 'B4', 'C5', 'D5', 'C5', 'A4', 'G4', 'C5', 'A4', '9.0.4', 'B4', 'A4', 'G4', 'E4', 'G4', 'A4', 'B4', 'C5', '9.0.4', 'B4']


In [ ]:
import numpy as np
from tensorflow.keras.utils import to_categorical

# Create a dictionary for each unique note/chord
unique_notes = sorted(set(notes))

note_to_int = {note: number for number, note in enumerate(unique_notes)}
int_to_note = {number: note for note, number in note_to_int.items()}

print("Unique notes/chords:", len(unique_notes))

# Create sequences of 50 notes
sequence_length = 50

network_input = []
network_output = []

for i in range(len(notes) - sequence_length):
    sequence = notes[i:i + sequence_length]
    next_note = notes[i + sequence_length]

    network_input.append([note_to_int[n] for n in sequence])
    network_output.append(note_to_int[next_note])

network_input = np.array(network_input)
network_output = np.array(network_output)

print("Input shape:", network_input.shape)
print("Output shape:", network_output.shape)

Unique notes/chords: 68
Input shape: (27405, 50)
Output shape: (27405,)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

# Reshape input for LSTM
network_input = network_input.reshape(
    network_input.shape[0],
    network_input.shape[1],
    1
)

# Normalize input
network_input = network_input / float(len(unique_notes))

# Convert output to categorical format
network_output = to_categorical(
    network_output,
    num_classes=len(unique_notes)
)

# Create LSTM model
model = Sequential([
    LSTM(128, input_shape=(network_input.shape[1], 1), return_sequences=True),
    Dropout(0.3),
    LSTM(128),
    Dropout(0.3),
    Dense(128, activation="relu"),
    Dense(len(unique_notes), activation="softmax")
])

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50, 128)        │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 68)             │         8,772 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 223,428 (872.77 KB)

 Trainable params: 223,428 (872.77 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    network_input,
    network_output,
    epochs=3,
    batch_size=256,
    validation_split=0.1
)

Epoch 1/3
97/97 ━━━━━━━━━━━━━━━━━━━━ 63s 644ms/step - accuracy: 0.1164 - loss: 3.0583 - val_accuracy: 0.0861 - val_loss: 3.1545
Epoch 2/3
97/97 ━━━━━━━━━━━━━━━━━━━━ 76s 585ms/step - accuracy: 0.1176 - loss: 3.0591 - val_accuracy: 0.0960 - val_loss: 3.1482
Epoch 3/3
97/97 ━━━━━━━━━━━━━━━━━━━━ 57s 588ms/step - accuracy: 0.1209 - loss: 3.0572 - val_accuracy: 0.0960 - val_loss: 3.1534


In [ ]:
import numpy as np

num_notes = 100

# Select a random starting sequence
start_index = np.random.randint(0, len(network_input))

# Use the original integer sequence before normalization
pattern = network_input[start_index].flatten().tolist()

generated_notes = []

for _ in range(num_notes):
    prediction_input = np.array(pattern[-50:]).reshape(1, 50, 1)

    # Predict the next note
    prediction = model.predict(prediction_input, verbose=0)

    # Select the most likely note
    index = np.argmax(prediction)

    # Convert number back to note/chord
    result = int_to_note[index]
    generated_notes.append(result)

    # Continue the sequence
    pattern.append(index)

print("Music generated successfully!")
print("Generated notes:", len(generated_notes))
print("First 20 notes:")
print(generated_notes[:20])

Music generated successfully!
Generated notes: 100
First 20 notes:
['A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4']


In [ ]:
from music21 import stream, note, chord

output = stream.Stream()

for item in generated_notes:
    try:
        # Chord
        if "." in item:
            chord_notes = [int(n) for n in item.split(".")]
            output.append(chord.Chord(chord_notes))
        else:
            # Single note
            output.append(note.Note(item))
    except:
        pass

# Save as MIDI
midi_path = "AI_Generated_Music.mid"
output.write("midi", fp=midi_path)

print("MIDI file created successfully!")
print("File:", midi_path)

MIDI file created successfully!
File: AI_Generated_Music.mid


In [ ]:
from google.colab import files

files.download("AI_Generated_Music.mid")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>